# 🧠 Neuro Camp Data Analysis Notebook

Welcome! In this notebook, you will analyze **real data** collected from neuroscience experiments — including fruit fly behavior, osmosis in potato tissue, and touch sensitivity across your own bodies.

You don't need to know how to code. Every code cell is already written for you. Your job is to:
1. **Run each cell** by clicking it and pressing `Shift + Enter`
2. **Read the output** — tables, charts, and statistics will appear below each cell
3. **Discuss the questions** with your group and TA

---

## How to use this notebook
- Cells with a gray background contain **code** — run them in order from top to bottom
- Cells with a white background (like this one) contain **text and questions**
- Never skip a cell — each one builds on the last

---

## ⚙️ Setup
Run this cell first. It loads the tools we need for the entire notebook.

In [ ]:
# Load libraries
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import tukey_hsd
import warnings
warnings.filterwarnings('ignore')

print('✅ Setup complete! You are ready to begin.')

---
# Part 1: 🦟 Drosophila Sleep & Activity

## Background

*Drosophila* (fruit flies) are one of the most important model organisms in neuroscience. Because their brains share many features with ours — including sleep circuits, circadian rhythms, and neurotransmitter systems — researchers use them to understand how the brain controls sleep and wakefulness.

In this dataset, researchers tracked **6 different species** of *Drosophila* over several hours, recording how much each fly slept and how active it was during each 30-minute window.

**Key question:** Do different species of *Drosophila* sleep and move differently from each other?

---

### Load the data

In [ ]:
# Load sleep and activity data
df_sleep = pd.read_csv('drosophila_sleep.csv', index_col=0)
df_activity = pd.read_csv('drosophila_activity.csv', index_col=0)

print('Sleep data: {} flies across {} species'.format(len(df_sleep), df_sleep['Species'].nunique()))
print('Activity data: {} flies across {} species'.format(len(df_activity), df_activity['Species'].nunique()))
print()
print('--- Preview: Sleep Data ---')
display(df_sleep.head(10))
print('--- Preview: Activity Data ---')
display(df_activity.head(10))

### Descriptive Statistics

In [ ]:
# Summary statistics for sleep
print('=== Average Sleep Per 30 Minutes by Species ===')
sleep_summary = df_sleep.groupby('Species')['Avg_Sleep_Per_30min'].agg(
    Count='count', Mean='mean', Std='std', Min='min', Max='max'
).round(2)
display(sleep_summary)

print()
print('=== Average Activity Per 30 Minutes by Species ===')
activity_summary = df_activity.groupby('Species')['Avg_Activity_Per_30min'].agg(
    Count='count', Mean='mean', Std='std', Min='min', Max='max'
).round(2)
display(activity_summary)

**💬 Discuss with your group and TA:**
1. Just from the table, which species sleeps the most? Which sleeps the least?
2. Which species is the most active? The least active?
3. Do you notice any pattern between sleep and activity across species?

### Bar Chart: Sleep by Species

In [ ]:
# Bar chart of average sleep per species
species_order = df_sleep.groupby('Species')['Avg_Sleep_Per_30min'].mean().sort_values(ascending=False).index

palette_sleep = sns.color_palette('colorblind', n_colors=df_sleep['Species'].nunique())

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(
    data=df_sleep,
    x='Species',
    y='Avg_Sleep_Per_30min',
    order=species_order,
    palette=palette_sleep,
    capsize=0.1,
    ax=ax
)
ax.set_title('Average Sleep Per 30 Minutes by Drosophila Species', fontsize=14, fontweight='bold')
ax.set_xlabel('Species', fontsize=12)
ax.set_ylabel('Average Sleep Per 30 Minutes (min)', fontsize=12)
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
sns.despine()
plt.tight_layout()
plt.show()

### Bar Chart: Activity by Species

In [ ]:
# Bar chart of average activity per species
species_order_act = df_activity.groupby('Species')['Avg_Activity_Per_30min'].mean().sort_values(ascending=False).index

palette_act = sns.color_palette('colorblind', n_colors=df_activity['Species'].nunique())

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(
    data=df_activity,
    x='Species',
    y='Avg_Activity_Per_30min',
    order=species_order_act,
    palette=palette_act,
    capsize=0.1,
    ax=ax
)
ax.set_title('Average Activity Per 30 Minutes by Drosophila Species', fontsize=14, fontweight='bold')
ax.set_xlabel('Species', fontsize=12)
ax.set_ylabel('Average Activity Per 30 Minutes (counts)', fontsize=12)
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
sns.despine()
plt.tight_layout()
plt.show()

**💬 Discuss with your group and TA:**
1. Does the ranking of species look the same in both charts, or does it change?
2. The error bars show the spread of individual fly measurements within each species. Which species has the most variability? What might that mean biologically?
3. Why might it matter for neuroscience research that different species have different sleep patterns?

### Statistical Test: One-Way ANOVA

The bar charts give us a visual sense of differences — but are they **statistically significant**? 

A **one-way ANOVA** tests whether the means of three or more groups are different from each other more than we'd expect by chance. It produces an **F-statistic** and a **p-value**.

- If **p < 0.05**, at least one group is significantly different from the others
- If **p ≥ 0.05**, we don't have enough evidence to say the groups differ

In [ ]:
# One-way ANOVA for sleep
sleep_groups = [group['Avg_Sleep_Per_30min'].values for _, group in df_sleep.groupby('Species')]
f_sleep, p_sleep = stats.f_oneway(*sleep_groups)

print('=== ANOVA: Sleep ===')
print(f'F-statistic: {f_sleep:.3f}')
print(f'p-value:     {p_sleep:.4f}')
if p_sleep < 0.05:
    print('✅ Result: Significant difference in sleep across species (p < 0.05)')
else:
    print('❌ Result: No significant difference in sleep across species (p ≥ 0.05)')

print()

# One-way ANOVA for activity
activity_groups = [group['Avg_Activity_Per_30min'].values for _, group in df_activity.groupby('Species')]
f_act, p_act = stats.f_oneway(*activity_groups)

print('=== ANOVA: Activity ===')
print(f'F-statistic: {f_act:.3f}')
print(f'p-value:     {p_act:.4f}')
if p_act < 0.05:
    print('✅ Result: Significant difference in activity across species (p < 0.05)')
else:
    print('❌ Result: No significant difference in activity across species (p ≥ 0.05)')

### Post-hoc Test: Tukey HSD

ANOVA tells us *that* groups differ, but not *which* pairs of species are different from each other. The **Tukey HSD** (Honestly Significant Difference) test compares every pair of species and tells us which ones are significantly different.

In [ ]:
# Tukey HSD for sleep
species_names_sleep = sorted(df_sleep['Species'].unique())
sleep_groups_ordered = [df_sleep[df_sleep['Species'] == s]['Avg_Sleep_Per_30min'].values for s in species_names_sleep]
tukey_sleep = tukey_hsd(*sleep_groups_ordered)

print('=== Tukey HSD: Sleep Pairwise Comparisons ===')
print(f'{"Species A":<20} {"Species B":<20} {"p-value":>10} {"Significant?":>14}')
print('-' * 68)
for i in range(len(species_names_sleep)):
    for j in range(i+1, len(species_names_sleep)):
        p = tukey_sleep.pvalue[i][j]
        sig = '✅ Yes' if p < 0.05 else 'No'
        print(f'{species_names_sleep[i]:<20} {species_names_sleep[j]:<20} {p:>10.4f} {sig:>14}')

In [ ]:
# Tukey HSD for activity
species_names_act = sorted(df_activity['Species'].unique())
activity_groups_ordered = [df_activity[df_activity['Species'] == s]['Avg_Activity_Per_30min'].values for s in species_names_act]
tukey_act = tukey_hsd(*activity_groups_ordered)

print('=== Tukey HSD: Activity Pairwise Comparisons ===')
print(f'{"Species A":<20} {"Species B":<20} {"p-value":>10} {"Significant?":>14}')
print('-' * 68)
for i in range(len(species_names_act)):
    for j in range(i+1, len(species_names_act)):
        p = tukey_act.pvalue[i][j]
        sig = '✅ Yes' if p < 0.05 else 'No'
        print(f'{species_names_act[i]:<20} {species_names_act[j]:<20} {p:>10.4f} {sig:>14}')

**💬 Discuss with your group and TA:**
1. Which pairs of species are significantly different in sleep? Does that match what you expected from the bar chart?
2. Which pairs are significantly different in activity?
3. *Drosophila* sleep is regulated by many of the same brain circuits as human sleep — including circuits involving dopamine and serotonin. Why might it be useful for neuroscientists to study sleep across different fly species rather than just one?
4. Can you think of a reason why two species might differ in activity but not sleep, or vice versa?

---
# Part 2: 🥔 Osmosis

## Background

**Osmosis** is the movement of water across a semi-permeable membrane from an area of low solute concentration to an area of high solute concentration. Every cell in your body depends on osmosis to maintain its size, shape, and function — including neurons.

In this experiment, potato pieces were placed in NaCl (salt) solutions of increasing concentration. By measuring the mass of each piece before and after soaking, we can calculate how much water moved in or out.

**Key question:** Does NaCl concentration significantly affect how much water moves into or out of potato tissue?

---

### Load and clean the data

In [ ]:
# Load osmosis data
df_osm = pd.read_csv('Osmosis_Class_Data.csv')

# Fill in group numbers (they were only listed once per group in the original sheet)
df_osm['Group #'] = df_osm['Group #'].ffill()

# Convert NaCl concentration from string ('0.9%') to number (0.9)
df_osm['NaCl Concentration (%)'] = df_osm['NaCl Concentration (%)'].str.replace('%', '').astype(float)

# Calculate percent mass change
df_osm['Percent Mass Change (%)'] = (
    (df_osm['Final Mass (g)'] - df_osm['Initial Mass (g)']) / df_osm['Initial Mass (g)'] * 100
).round(2)

print('Osmosis data loaded: {} observations across {} groups and {} concentrations'.format(
    len(df_osm), int(df_osm['Group #'].nunique()), df_osm['NaCl Concentration (%)'].nunique()
))
display(df_osm)

### Descriptive Statistics

In [ ]:
# Average percent mass change per concentration
osm_summary = df_osm.groupby('NaCl Concentration (%)')['Percent Mass Change (%)'].agg(
    Count='count', Mean='mean', Std='std', Min='min', Max='max'
).round(2)

print('=== Percent Mass Change by NaCl Concentration ===')
display(osm_summary)

**💬 Discuss with your group and TA:**
1. At 0% NaCl (pure water), did the potato gain or lose mass? Why?
2. At 8% NaCl, did the potato gain or lose mass? Why?
3. Based on the table, roughly what concentration do you think is closest to the potato's own internal salt concentration? How can you tell?

### Line Graph: Percent Mass Change vs. NaCl Concentration

In [ ]:
# Line graph of percent mass change vs. concentration
conc_means = df_osm.groupby('NaCl Concentration (%)')['Percent Mass Change (%)'].mean().reset_index()

fig, ax = plt.subplots(figsize=(9, 6))

# Plot individual group points
sns.scatterplot(
    data=df_osm,
    x='NaCl Concentration (%)',
    y='Percent Mass Change (%)',
    hue='Group #',
    palette='colorblind',
    s=80,
    zorder=3,
    ax=ax,
    legend=True
)

# Plot mean line
ax.plot(
    conc_means['NaCl Concentration (%)'],
    conc_means['Percent Mass Change (%)'],
    color='black',
    linewidth=2.5,
    marker='D',
    markersize=8,
    label='Class Mean',
    zorder=4
)

# Reference line at 0
ax.axhline(0, color='gray', linestyle='--', linewidth=1.2, label='No change')

ax.set_title('Percent Mass Change of Potato vs. NaCl Concentration', fontsize=14, fontweight='bold')
ax.set_xlabel('NaCl Concentration (%)', fontsize=12)
ax.set_ylabel('Percent Mass Change (%)', fontsize=12)
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, labels, title='Group / Line', bbox_to_anchor=(1.02, 1), loc='upper left')
sns.despine()
plt.tight_layout()
plt.show()

**💬 Discuss with your group and TA:**
1. Describe the trend you see in the line. As salt concentration increases, what happens to mass change?
2. Where does the black line (class mean) cross zero? What does that crossing point represent biologically?
3. Do all groups show the same pattern, or is there variability? What could cause differences between groups?
4. **Neuroscience connection:** The brain is surrounded by cerebrospinal fluid (CSF). If the salt concentration of CSF changes — for example, due to severe dehydration or overhydration — what might happen to neurons based on what you observed with the potato?

### Statistical Test: One-Way ANOVA

In [ ]:
# One-way ANOVA across NaCl concentrations
osm_groups = [group['Percent Mass Change (%)'].values for _, group in df_osm.groupby('NaCl Concentration (%)')]
f_osm, p_osm = stats.f_oneway(*osm_groups)

print('=== ANOVA: Osmosis ===')
print(f'F-statistic: {f_osm:.3f}')
print(f'p-value:     {p_osm:.4f}')
if p_osm < 0.05:
    print('✅ Result: Significant difference in mass change across concentrations (p < 0.05)')
else:
    print('❌ Result: No significant difference in mass change across concentrations (p ≥ 0.05)')

### Post-hoc Test: Tukey HSD

In [ ]:
# Tukey HSD for osmosis
conc_labels = sorted(df_osm['NaCl Concentration (%)'].unique())
osm_groups_ordered = [df_osm[df_osm['NaCl Concentration (%)'] == c]['Percent Mass Change (%)'].values for c in conc_labels]
tukey_osm = tukey_hsd(*osm_groups_ordered)

print('=== Tukey HSD: Osmosis Pairwise Comparisons ===')
print(f'{"NaCl % (A)":>12} {"NaCl % (B)":>12} {"p-value":>10} {"Significant?":>14}')
print('-' * 52)
for i in range(len(conc_labels)):
    for j in range(i+1, len(conc_labels)):
        p = tukey_osm.pvalue[i][j]
        sig = '✅ Yes' if p < 0.05 else 'No'
        print(f'{conc_labels[i]:>12} {conc_labels[j]:>12} {p:>10.4f} {sig:>14}')

**💬 Discuss with your group and TA:**
1. Which concentration pairs are significantly different from each other?
2. Are the high-concentration comparisons (e.g., 4% vs 8%) significant? Why or why not?
3. The ANOVA and Tukey test assume that concentration *causes* mass change. Is that a fair assumption here? What would make you more or less confident in that claim?

---
# Part 3: ✋ Two-Point Discrimination

## Background

**Two-point discrimination** is the ability to detect that two nearby points touching the skin are two separate points — not one. The smallest distance at which you can tell them apart is called the **discrimination threshold**.

This threshold varies across the body because different skin regions have different **densities of touch receptors** (called mechanoreceptors). Areas with more receptors packed closely together — like fingertips — have finer spatial resolution and lower thresholds. Areas with fewer receptors — like the upper arm — have coarser resolution and higher thresholds.

In this dataset, each of you measured your own two-point discrimination threshold (in mm) at 6 body sites.

**Key question:** Does two-point discrimination threshold differ significantly across body sites?

---

### Load the data

In [ ]:
# Load two-point discrimination data
df_tp = pd.read_excel('Two_point_discrimination.xlsx')

print('Two-point discrimination data: {} students, {} body sites'.format(
    len(df_tp), len(df_tp.columns) - 1  # subtract Student column
))
display(df_tp)

### Descriptive Statistics

In [ ]:
# Calculate mean and SD for each body site
body_sites = ['Wrist', 'Palm', 'Back of Hand', 'Thumb', 'Cheek', 'Upper Arm']

tp_summary = df_tp[body_sites].agg(['count', 'mean', 'std', 'min', 'max']).T.round(2)
tp_summary.columns = ['Count', 'Mean (mm)', 'Std (mm)', 'Min (mm)', 'Max (mm)']
tp_summary = tp_summary.sort_values('Mean (mm)')

print('=== Two-Point Discrimination Threshold by Body Site (mm) ===')
display(tp_summary)

**💬 Discuss with your group and TA:**
1. Which body site has the lowest threshold (finest touch sensitivity)? Which has the highest?
2. Does this match what you would predict based on your everyday experience with touch?
3. Why might the thumb have a lower threshold than the wrist?

### Bar Chart: Mean Threshold by Body Site

In [ ]:
# Reshape data to long format for seaborn
df_tp_long = df_tp.melt(id_vars='Student', value_vars=body_sites, var_name='Body Site', value_name='Threshold (mm)')

# Order by mean threshold
site_order = df_tp[body_sites].mean().sort_values().index.tolist()
palette_tp = sns.color_palette('colorblind', n_colors=len(body_sites))

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(
    data=df_tp_long,
    x='Body Site',
    y='Threshold (mm)',
    order=site_order,
    palette=palette_tp,
    capsize=0.1,
    ax=ax
)
ax.set_title('Two-Point Discrimination Threshold by Body Site', fontsize=14, fontweight='bold')
ax.set_xlabel('Body Site', fontsize=12)
ax.set_ylabel('Mean Threshold (mm)', fontsize=12)
sns.despine()
plt.tight_layout()
plt.show()

**💬 Discuss with your group and TA:**
1. Look at the error bars — which body sites have the most variability across students? What might explain that?
2. **Neuroscience connection:** In the brain, there is a map of the body called the **somatosensory homunculus**. Body parts with lower thresholds (finer touch) take up more space in the brain map. Based on your chart, which body parts would you expect to have the largest representation in the brain?
3. How might this relate to the spinal cord cross-section you saw in dissection — specifically the dorsal horn, which receives touch signals from the body?

### Statistical Test: One-Way ANOVA

In [ ]:
# One-way ANOVA across body sites
tp_groups = [df_tp[site].values for site in body_sites]
f_tp, p_tp = stats.f_oneway(*tp_groups)

print('=== ANOVA: Two-Point Discrimination ===')
print(f'F-statistic: {f_tp:.3f}')
print(f'p-value:     {p_tp:.4f}')
if p_tp < 0.05:
    print('✅ Result: Significant difference in thresholds across body sites (p < 0.05)')
else:
    print('❌ Result: No significant difference in thresholds across body sites (p ≥ 0.05)')

### Post-hoc Test: Tukey HSD

In [ ]:
# Tukey HSD for two-point discrimination
tukey_tp = tukey_hsd(*tp_groups)

print('=== Tukey HSD: Two-Point Discrimination Pairwise Comparisons ===')
print(f'{"Site A":<16} {"Site B":<16} {"p-value":>10} {"Significant?":>14}')
print('-' * 60)
for i in range(len(body_sites)):
    for j in range(i+1, len(body_sites)):
        p = tukey_tp.pvalue[i][j]
        sig = '✅ Yes' if p < 0.05 else 'No'
        print(f'{body_sites[i]:<16} {body_sites[j]:<16} {p:>10.4f} {sig:>14}')

**💬 Discuss with your group and TA:**
1. Which pairs of body sites are significantly different from each other?
2. Are there any pairs you expected to be different that weren't? Why might that be?
3. The data came from your own class. How might results differ if we measured a different group — for example, professional musicians, or people who work with their hands?
4. **Big picture:** All three of today's datasets — fly sleep, osmosis, and two-point discrimination — involved measuring biological variation. What is one thing these experiments have in common in terms of how we analyzed the data?

---
# 🎉 You're done!

Today you:
- Loaded and explored **real neuroscience data**
- Created **publication-style bar charts and line graphs**
- Ran **one-way ANOVAs** to test for group differences
- Used **Tukey HSD** to identify which specific groups differed
- Connected your findings to **neuroscience concepts** including sleep circuits, osmotic balance in the brain, and the somatosensory homunculus

These are the same tools that researchers use every day. Nice work! 🧠